In [1]:
import os
import torch
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew, Process
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from crewai.tools import tool

load_dotenv()

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize LLMs (Using Groq as per your LegalRAG2 setup)
legal_llm = LLM(model="groq/llama-3.3-70b-versatile")
research_llm = LLM(model="groq/llama-3.1-8b-instant")

# Set dummy key to satisfy any internal Pydantic checks (Method from Lawglance)
os.environ["OPENAI_API_KEY"] = "NA"

In [2]:
from transformers import AutoModel, AutoTokenizer
import torch
import numpy as np
from sklearn.cluster import KMeans

# We will build a lightweight manual extractor to avoid the ImportError
class SimpleBertSummarizer:
    def __init__(self):
        self.model_name = "bert-base-uncased"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name)
        self.model.eval()

    def __call__(self, text, num_sentences=2):
        sentences = text.split('. ')
        if len(sentences) <= num_sentences:
            return text

        # Get embeddings for each sentence
        inputs = self.tokenizer(sentences, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        # Use the [CLS] token representation for each sentence
        embeddings = outputs.last_hidden_state[:, 0, :].numpy()

        # Cluster sentences and pick the ones closest to the center (most representative)
        kmeans = KMeans(n_clusters=num_sentences, n_init=10).fit(embeddings)
        avg = []
        for i in range(num_sentences):
            idx = np.argmin(np.linalg.norm(embeddings - kmeans.cluster_centers_[i], axis=1))
            avg.append(idx)
        
        return ". ".join([sentences[i] for i in sorted(avg)])

# Initialize this instead of the broken Summarizer()
bert_model = SimpleBertSummarizer()

def compress_agent_output(task_output):
    raw_text = task_output.raw
    if len(raw_text.split()) < 50:
        return raw_text
    
    compressed_text = bert_model(raw_text, num_sentences=2)
    print(f"\n--- [BERT COMPRESSION SUCCESSFUL] ---")
    return compressed_text

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
import re
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
# Initialize once outside the tool to save time
# Force CPU to ensure it doesn't compete with your embeddings for GPU memory


# 1. Initialize Embeddings
# Direct LangChain logic from your working Lawglance files
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={'device': device}
)

# 2. Connect to your existing Vector Database
# Update this path to your actual vector_database folder
db_path = "C:/Users/user/Desktop/RAG Projects/Legal RAG 1/vector_database"
vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)

def compress_legal_output(text, num_sentences=4): # Increased to 4 for better context
    if not text: return ""
    
    # 1. BETTER SPLITTING: Don't split on dots in 'Sec.' or 'v.'
    # This regex looks for periods followed by a space and a capital letter
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 25]
    
    if len(sentences) <= num_sentences:
        return text

    # 2. LEGAL WEIGHTING: Identify sentences with legal citations
    # We want to force-include sentences that mention "Section", "Article", or "Act"
    legal_keywords = r"Section|Article|Act|BNS|IPC|Constitution|Clause|u/s|v\."
    important_indices = [i for i, s in enumerate(sentences) if re.search(legal_keywords, s, re.IGNORECASE)]
    
    # 3. EMBEDDING & CLUSTERING (as before)
    sent_embeddings = embeddings.embed_documents(sentences)
    kmeans = KMeans(n_clusters=num_sentences, random_state=42, n_init=10).fit(sent_embeddings)
    
    avg = []
    for center in kmeans.cluster_centers_:
        distances = cdist([center], sent_embeddings, 'euclidean')
        avg.append(np.argmin(distances))
    
    # 4. UNION: Combine the 'Centroids' and the 'Legal Citation' sentences
    # This ensures we get both the general context AND the specific law
    final_indices = sorted(list(set(avg) | set(important_indices[:2]))) # Keep top 2 legal markers
    
    summary = " ".join([sentences[i] for i in final_indices])
    return summary

# 3. Create the Custom Tool function
@tool("legal_research_tool")
def legal_research_tool(query: str):
    """Searches the database and compresses the legal output to save tokens."""
    # 1. Retrieve raw chunks
    docs = vector_store.similarity_search(query, k=8)
    raw_content = "\n\n".join([d.page_content for d in docs])
    
    # 2. Compress using our custom BERT-logic
    # This reduces a 1000-word block into the 3 most important legal sentences
    compressed_content = compress_legal_output(raw_content, num_sentences=3)
    
    return str(compressed_content)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Agent 1: The Researcher
researcher = Agent(
    role='Legal Researcher',
    goal='Retrieve and present the specific legal citations and summaries provided by the research tool for {user_query}',
    # Shortened for token efficiency
    backstory='Expert in Indian Constitutional law and criminal statutes.', 
    tools=[legal_research_tool],
    llm=research_llm,
    max_rpm=1, # Key for Groq Free Tier
    verbose=True
)

# Agent 2: The Strategist
strategist = Agent(
    role='Legal Strategist',
    goal='Determine legal hierarchy (e.g., Central vs. State law) from findings',
    backstory='Senior counsel specializing in jurisdictional conflicts.',
    llm=legal_llm,
    max_rpm=1,
    verbose=True
)

# Agent 3: The Advisor
advisor = Agent(
    role='Legal Advisor',
    goal='Synthesize a final, easy-to-understand response for the user',
    backstory='Advisor who translates complex legalese for citizens.',
    llm=legal_llm,
    max_rpm=1,
    verbose=True
)

In [5]:
research_task = Task(
    description='Find 2 legal provisions for {user_query}. Do not quote full text, just the core rule.',
    expected_output='A summary containing the relevant Section/Article numbers and the core legal rule.',
    agent=researcher,
    callback=compress_agent_output # <--- BERT runs immediately after this task
)

strategy_task = Task(
    description='Analyze the findings. If laws conflict, specify which one takes precedence.',
    expected_output='A strategic determination of the primary applicable law.',
    agent=strategist,
    context=[research_task] # Strategist only receives the COMPRESSED output
)

advisory_task = Task(
    description='Give a short, actionable answer to the user.',
    expected_output='A final legal response under 50 words.',
    agent=advisor
)

In [6]:
import time

legal_crew = Crew(
    agents=[researcher, strategist, advisor],
    tasks=[research_task, strategy_task, advisory_task],
    process=Process.sequential,
    memory=False, # Mandatory: Keeps internal TPM usage low
    verbose=True,
    max_rpm=2 # Mandatory for Groq
)

In [7]:
# Question 1
user_input = "I was caught jumping a red light and the police officer is asking for a 5,000 rupee fine. Is this amount correct?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Motor Vehicles Act' in result.raw else 'FAIL'}")
print(f"Reasoning: Check if the AI corrected the fine amount based on the database.")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  6ab13af3-6100-4406-a68a-efb68ed72e45                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find 2 legal provisions for I was caught jumping a red light and the police officer is asking for a      │
│  5,000 rupee fine. Is this amount correct?. Do not quote full text, just the core rule.                         │
│  ID: f474aa89-4cf6-4f55-aee5-c44318297fbc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Find 2 legal provisions for I was caught jumping a red light and the police officer is asking for a      │
│  5,000 rupee fine. Is this amount correct?. Do not quote full text, just the core rule.                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': 'Jumping a red light 5000 rupee fine, is it correct according to the Indian laws? Show legal   │
│  citations and summaries.'}                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Give a short, actionable answer to the user.                                                                   │
│  Agent:                                                                                                         │
│  Legal Advisor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  6ab13af3-6100-4406-a68a-efb68ed72e45                                                                           │
│  Final Output: The Motor Vehicles Act, 1988, applies, with fines determined by Sections 206, 207, 208, 202,     │
│  and 203.                                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Evaluation: PASS
Reasoning: Check if the AI corrected the fine amount based on the database.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# Question 5
user_input = "Police searched phone/WhatsApp without a warrant for protest evidence. Does this violate Privacy laws?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Privacy' in result.raw or 'Puttaswamy' in result.raw else 'FAIL'}")
print("Check: Did it mention that even for national security, 'procedure established by law' must be followed?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  6ab13af3-6100-4406-a68a-efb68ed72e45                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find 2 legal provisions for Police searched phone/WhatsApp without a warrant for protest evidence. Does  │
│  this violate Privacy laws?. Do not quote full text, just the core rule.                                        │
│  ID: f474aa89-4cf6-4f55-aee5-c44318297fbc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Find 2 legal provisions for Police searched phone/WhatsApp without a warrant for protest evidence. Does  │
│  this violate Privacy laws?. Do not quote full text, just the core rule.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': 'Police searched phone/WhatsApp without a warrant for protest evidence. Is it violating the    │
│  Privacy laws? What are relevant Section/Article numbers?'}                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Give a short, actionable answer to the user.                                                                   │
│  Agent:                                                                                                         │
│  Legal Advisor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  6ab13af3-6100-4406-a68a-efb68ed72e45                                                                           │
│  Final Output: Police need a warrant to search phones/WhatsApp, unless exceptions apply under Section 93(1) of  │
│  the CrPC, protecting right to privacy under Article 21.                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Evaluation: FAIL
Check: Did it mention that even for national security, 'procedure established by law' must be followed?


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯